# 01 - Regressão Linear

## Por que usar este modelo?

Use Regressão Linear quando você quer prever uma resposta quantitativa e precisa de um modelo simples, rápido e interpretável.

Exemplos comuns:
* estimar preço, demanda, risco ou tempo de entrega;
* medir quanto a resposta média muda quando um preditor aumenta;
* criar uma baseline antes de modelos mais complexos.

Seguindo a notação do ISLP, temos uma resposta ($Y$) e ($p$) preditores ($X_1, X_2, \dots, X_p$). Para a observação ($i$), o modelo linear múltiplo é:

$$Y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_p x_{ip} + \epsilon_i$$

A previsão estimada é:

$$\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_{i1} + \hat{\beta}_2 x_{i2} + \dots + \hat{\beta}_p x_{ip}$$

Onde:
* ($i = 1, \dots, n$): índice da observação no dataset.
* ($j = 1, \dots, p$): índice do preditor.
* ($Y_i$): valor real da resposta para a observação ($i$).
* ($x_{ij}$): valor do preditor ($j$) na observação ($i$).
* ($\beta_0$): intercepto, isto é, valor esperado de ($Y$) quando todos os preditores são zero.
* ($\beta_j$): efeito médio associado ao preditor ($X_j$), mantendo os demais constantes.
* ($\epsilon_i$): erro irredutível, isto é, a parte de ($Y_i$) não explicada pelos preditores.
* ($\hat{\beta}_j$): estimativa aprendida a partir dos dados.
* ($\hat{y}_i$): valor previsto pelo modelo para a observação ($i$).

Os resíduos são:

$$e_i = y_i - \hat{y}_i$$

Onde ($e_i$) é o erro observado após ajustar o modelo. O ($R^2$) mede a fração da variabilidade de ($Y$) explicada pelo modelo:

$$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}$$

Onde ($\bar{y}$) é a média dos valores reais da resposta no conjunto avaliado.

In [ ]:
# Importa numpy para operacoes numericas e criacao de grades de valores.
import numpy as np

# Importa pandas para organizar dados tabulares em DataFrames.
import pandas as pd

# Importa matplotlib para graficos base.
import matplotlib.pyplot as plt

# Importa seaborn para graficos estatisticos com estetica melhor.
import seaborn as sns

# Carrega o dataset diabetes, nativo do scikit-learn.
from sklearn.datasets import load_diabetes

# Divide os dados em treino e teste.
from sklearn.model_selection import train_test_split

# Modelo de Regressao Linear do scikit-learn.
from sklearn.linear_model import LinearRegression

# Metricas para avaliar erro e poder explicativo.
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Define um tema visual consistente para os graficos.
sns.set_theme(style="whitegrid", context="notebook")

## Carregando os dados

Usamos o dataset `diabetes` do scikit-learn tanto para a regressão simples quanto para a múltipla. O alvo representa a progressão da doença um ano após a linha de base.

In [ ]:
# Carrega o dataset diabetes como um objeto tipo dicionario.
diabetes = load_diabetes()

In [ ]:
# Mostra as caracteristicas do dataset
print(diabetes.DESCR)

In [ ]:
# Converte as features para DataFrame e preserva os nomes das colunas.
X_all = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
X_all.head()

In [ ]:
# Cria a variavel alvo como Series; ela representa progressao da doenca.
y = pd.Series(diabetes.target, name="target")
y.head()

## Implementação: regressão simples

Na regressão linear simples, usamos apenas um preditor:

$$Y_i = \beta_0 + \beta_1 x_i + \epsilon_i$$

Onde ($x_i$) é o valor do único preditor da observação ($i$). Aqui ele será `bmi`. O modelo aprende ($\hat{\beta}_0$) e ($\hat{\beta}_1$) por mínimos quadrados, isto é, escolhendo os coeficientes que minimizam:

$$RSS = \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

No ISLP, esse termo aparece como *Residual Sum of Squares*. Quanto menor o RSS no treino, melhor a reta se ajusta aos pontos de treino; mas isso não garante boa generalização.

In [ ]:
# Seleciona apenas a coluna bmi para demonstrar regressao linear simples.
X_simple = X_all[["bmi"]]
X_simple.head()

In [ ]:
# Divide dados em treino e teste; test_size controla a fracao reservada para avaliacao.
X_train, X_test, y_train, y_test = train_test_split(
    X_simple,  # Features usadas pelo modelo.
    y,         # Valores reais que queremos prever.
    test_size=0.25,     # 25% dos dados ficam fora do treino.
    random_state=42,    # Garante reprodutibilidade da divisao.
)

# Mostra o shape dos conjuntos de treino e teste.
print(f"X_simple, y shape: {X_simple.shape}, {y.shape}")
print(f"X_train, y_train shape: {X_train.shape}, {y_train.shape}")
print(f"X_test, y_test shape: {X_test.shape}, {y_test.shape}")

# Mostra os primeiros registros do conjunto de treino.
X_train.head()

In [ ]:
# Cria o modelo; fit_intercept=True aprende o beta_0 da reta.
simple_model = LinearRegression(fit_intercept=True)

# Ajusta beta_0 e beta_1 minimizando a soma dos residuos quadraticos no treino.
simple_model.fit(X_train, y_train)

In [ ]:
# Gera previsoes para os exemplos de teste.
y_pred_simple = simple_model.predict(X_test)

# Calcula residuos: diferenca entre valor real e valor previsto.
residuals = y_test - y_pred_simple

# Calcula MAE: erro medio em unidades originais do alvo.
mae = mean_absolute_error(y_test, y_pred_simple)

# Calcula RMSE: penaliza erros grandes por usar erro quadratico.
rmse = root_mean_squared_error(y_test, y_pred_simple)

# Calcula R2: proporcao da variancia explicada pelo modelo.
r2 = r2_score(y_test, y_pred_simple)

# Mostra coeficiente angular, intercepto e metricas principais.
print(f"Intercepto beta_0: {simple_model.intercept_:.2f}")
print(f"Coeficiente beta_1 para bmi: {simple_model.coef_[0]:.2f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

In [ ]:
# Cria uma sequencia de valores de bmi para desenhar a reta aprendida.
x_line = np.linspace(X_simple["bmi"].min(), X_simple["bmi"].max(), 100).reshape(-1, 1)

# Calcula a previsao do modelo para cada ponto da reta.
y_line = simple_model.predict(x_line)

# Define o tamanho da figura.
plt.figure(figsize=(9, 6))

# Plota os dados reais como pontos.
plt.scatter(X_simple["bmi"], y, alpha=0.55, label="Dados reais")

# Plota a reta de regressao aprendida.
plt.plot(x_line, y_line, color="crimson", linewidth=3, label="Reta ajustada")

# Nomeia o eixo x.
plt.xlabel("bmi")

# Nomeia o eixo y.
plt.ylabel("Progressão da doença")

# Adiciona titulo explicativo.
plt.title("Regressão Linear Simples: ajuste de uma reta")

# Exibe a legenda.
plt.legend()

# Mostra o grafico.
plt.show()

In [ ]:
# Cria figura com dois graficos lado a lado.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plota valores reais contra previstos; idealmente ficam proximos da diagonal.
axes[0].scatter(y_test, y_pred_simple, alpha=0.7)

# Cria limites comuns para desenhar a diagonal perfeita.
limits = [min(y_test.min(), y_pred_simple.min()), max(y_test.max(), y_pred_simple.max())]

# Desenha linha y=x, que representa previsao perfeita.
axes[0].plot(limits, limits, linestyle="dashed", color="lightgrey")

# Nomeia o eixo x do primeiro grafico.
axes[0].set_xlabel("Valor real")

# Nomeia o eixo y do primeiro grafico.
axes[0].set_ylabel("Valor previsto")

# Titulo do primeiro grafico.
axes[0].set_title("Real vs. previsto")

# Plota distribuicao dos residuos para verificar erros sistematicos.
sns.histplot(residuals, kde=True, ax=axes[1], color="steelblue")

# Desenha linha vertical em zero, onde nao ha erro.
axes[1].axvline(0, color="crimson", linewidth=2)

# Nomeia o eixo x do segundo grafico.
axes[1].set_xlabel("Resíduo = real - previsto")

# Titulo do segundo grafico.
axes[1].set_title("Distribuição dos resíduos")

# Ajusta espacamentos.
plt.tight_layout()

# Mostra os graficos.
plt.show()

## Como interpretar os resultados?

A regressão linear simples com `bmi` mostrou que existe relação linear entre o preditor e a progressão da doença, mas com capacidade explicativa limitada por usar apenas uma variável.

* **Coeficiente ($\beta_1$):** indica o efeito médio do `bmi` na resposta. Como observado no ajuste, o sinal do coeficiente mostra a direção dessa relação (neste caso, aumento de `bmi` tende a elevar a predição).
* **Intercepto ($\beta_0$):** é o valor base previsto quando `bmi` = 0 (na escala transformada do dataset).
* **MAE e RMSE:** quantificam o erro de previsão em unidades do alvo; quanto menores, melhor. Se o RMSE fica acima do MAE, há indício de alguns erros maiores (outliers).
* **$R^2$:** mede a fração da variabilidade explicada pelo modelo. Aqui ele sugere que o modelo captura parte do padrão, mas ainda deixa bastante variação sem explicação.
* **Diagnóstico visual:**
  * No gráfico real vs. previsto, a dispersão em torno da diagonal indica erro relevante.
  * Na distribuição de resíduos, a concentração perto de zero é desejável; espalhamento amplo indica limitação do modelo simples.

**Conclusão:** o modelo funciona como baseline interpretável, mas para ganho de desempenho a próxima etapa é usar regressão múltipla com mais preditores (e, se necessário, regularização).

---

### Report executivo (negócio)

* **Objetivo:** estimar a progressão da doença a partir do `bmi` com um modelo simples e interpretável.
* **Resultado principal:** o `bmi` tem relação positiva com a progressão (quando aumenta, a previsão tende a aumentar), confirmando sinal estatístico útil.
* **Qualidade preditiva:** desempenho moderado, adequado como baseline, mas insuficiente para decisões críticas isoladas.
* **Leitura para decisão:** o modelo é bom para explicação inicial e priorização exploratória, não para previsão final de alta precisão.
* **Próximos passos recomendados:** evoluir para regressão múltipla com mais variáveis clínicas, comparar métricas em validação e avaliar regularização para ganho de estabilidade e acurácia.

## Checkpoint para o Engenheiro

Pense na Regressão Linear como uma função de scoring com pesos fixos:

$$\text{score} = \text{bias} + w_1 \cdot \text{feature}_1 + w_2 \cdot \text{feature}_2 + \dots$$

Ela se parece com uma regra de negócio parametrizada, só que os pesos são aprendidos automaticamente a partir dos dados.

Em outras palavras, a Regressão Linear tenta encontrar os melhores valores para $\text{bias}$ e $w_1, w_2, \dots$ de forma que o $\text{score}$ se aproxime o máximo possível do valor real que queremos prever. Isso é feito minimizando uma função de perda, geralmente o erro quadrático médio (MSE), que mede a diferença entre os valores previstos ($\text{score}$) e os valores reais.

## Implementação: regressão múltipla

Agora usamos todos os ($p$) preditores disponíveis. Em forma matricial, uma maneira compacta de escrever o modelo é:

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\epsilon}$$

Onde:
* ($\mathbf{y}$): vetor ($n \times 1$) com as respostas reais.
* ($\mathbf{X}$): matriz ($n \times (p + 1)$) com uma coluna de 1s para o intercepto e ($p$) colunas de preditores.
* ($\boldsymbol{\beta}$): vetor ($(p + 1) \times 1$) com intercepto e coeficientes.
* ($\boldsymbol{\epsilon}$): vetor de erros.

Abrindo a multiplicação matricial, fica visível o que cada bloco representa — a coluna de 1s multiplica o intercepto, e cada coluna de features multiplica seu respectivo parâmetro:

$$\begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{bmatrix}_{(n \times 1)} = \begin{bmatrix} 1 & x_{11} & x_{12} & \dots & x_{1p} \\ 1 & x_{21} & x_{22} & \dots & x_{2p} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_{n1} & x_{n2} & \dots & x_{np} \end{bmatrix}_{(n \times (p+1))} \begin{bmatrix} \beta_0 \\ \beta_1 \\ \beta_2 \\ \vdots \\ \beta_p \end{bmatrix}_{((p+1) \times 1)} + \begin{bmatrix} \epsilon_1 \\ \epsilon_2 \\ \vdots \\ \epsilon_n \end{bmatrix}_{(n \times 1)}$$

Multiplicando a linha ($i$) de ($\mathbf{X}$) pelo vetor ($\boldsymbol{\beta}$), recuperamos exatamente a soma ponderada de uma única observação:

$$y_i = \beta_0 \cdot 1 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_p x_{ip} + \epsilon_i$$

Ou seja: cada linha de ($\mathbf{X}$) é uma observação, cada coluna é uma feature (a primeira, de 1s, ativa o intercepto), e o vetor ($\boldsymbol{\beta}$) guarda os parâmetros aprendidos — um peso por feature, mais o intercepto.

## Como o modelo é \"treinado\"?

Treinar significa encontrar o vetor ($\boldsymbol{\beta}$) que minimiza a soma dos resíduos quadráticos (RSS), agora escrita em forma matricial:

$$RSS(\boldsymbol{\beta}) = \Vert{}\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\Vert{}_2^2 = (\mathbf{y} - \mathbf{X}\boldsymbol{\beta})^T (\mathbf{y} - \mathbf{X}\boldsymbol{\beta})$$

### 1) Solução analítica (equações normais)

Como o RSS é uma função quadrática e convexa em ($\boldsymbol{\beta}$), o mínimo está no ponto onde o gradiente se anula. Derivando em relação a ($\boldsymbol{\beta}$) e igualando a zero:

$$\frac{\partial RSS}{\partial \boldsymbol{\beta}} = -2\mathbf{X}^T (\mathbf{y} - \mathbf{X}\boldsymbol{\beta}) = 0$$

que reorganizado nos dá as **equações normais**:

$$\mathbf{X}^T \mathbf{X}\boldsymbol{\beta} = \mathbf{X}^T \mathbf{y}$$

Se ($\mathbf{X}^T \mathbf{X}$) for invertível (posto completo das colunas), a solução fechada é:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

Onde:
* ($\mathbf{X}^T \mathbf{X}$): matriz ($(p + 1) \times (p + 1)$) (proporcional à matriz de covariância dos preditores).
* ($\mathbf{X}^T \mathbf{y}$): vetor ($(p + 1) \times 1$) que correlaciona cada preditor com o alvo.

Essa fórmula é exata, mas raramente é usada diretamente na prática. Inverter ($\mathbf{X}^T \mathbf{X}$) custa ($\mathcal{O}(p^3)$) e, sobretudo, é **numericamente instável**: elevar ao quadrado a matriz ($\mathbf{X}$) dobra o número de condição, amplificando erros de arredondamento quando há colinearidade entre features.

### 2) Solução numérica (SVD) — a usada pelo scikit-learn

Para `LinearRegression`, o scikit-learn (via `scipy.linalg.lstsq`, que chama LAPACK) não monta as equações normais. Em vez disso, resolve o problema de mínimos quadrados diretamente sobre ($\mathbf{X}$) usando a **decomposição em valores singulares (SVD)**:

$$\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

Onde:
* ($\mathbf{U}$): matriz ($n \times n$) ortogonal (vetores singulares à esquerda).
* ($\boldsymbol{\Sigma}$): matriz ($n \times (p + 1)$) diagonal com os valores singulares ($\sigma_1 \ge \sigma_2 \ge \dots \ge 0$).
* ($\mathbf{V}$): matriz ($(p + 1) \times (p + 1)$) ortogonal (vetores singulares à direita).

Substituindo a SVD na solução, a estimativa vira:

$$\hat{\boldsymbol{\beta}} = \mathbf{V} \boldsymbol{\Sigma}^\dagger \mathbf{U}^T \mathbf{y} = \mathbf{X}^\dagger \mathbf{y}$$

Onde ($\mathbf{X}^\dagger = \mathbf{V} \boldsymbol{\Sigma}^\dagger \mathbf{U}^T$) é a **pseudo-inversa de Moore-Penrose**. O termo ($\boldsymbol{\Sigma}^\dagger$) é obtido invertendo cada valor singular não nulo ($1/\sigma_i$) e transpondo a matriz; valores singulares abaixo de uma tolerância são tratados como zero.

**Vantagens dessa abordagem sobre as equações normais:**
* **Estabilidade numérica:** trabalha com o número de condição de ($\mathbf{X}$), e não com o de ($\mathbf{X}^T \mathbf{X}$) (que é o quadrado do primeiro).
* **Robustez à colinearidade / posto incompleto:** quando ($\mathbf{X}^T \mathbf{X}$) é singular, a inversa não existe, mas a pseudo-inversa sempre existe e devolve a solução de menor norma dentre as infinitas que minimizam o RSS.

Em resumo: a solução analítica com equações normais explica o que está sendo calculado; a SVD é como isso é calculado de forma confiável nos pacotes computacionais. Ambas retornam o mesmo ($\hat{\boldsymbol{\beta}}$) quando ($\mathbf{X}^T \mathbf{X}$) é bem-condicionada.

## Exercício

Refaça o fluxo de treinamento e avaliação da regressão linear, agora usando todas as features disponíveis no dataset de diabetes.

### Objetivo
Construir um modelo de Regressão Linear Múltipla com todos os preditores e avaliar sua qualidade preditiva em treino e teste.

### Interpretação esperada
* Quais variáveis parecem ter maior impacto positivo e negativo no alvo?
* O modelo generaliza bem do treino para o teste?
* Há sinais de erro sistemático (viés) nos resíduos?
* O nível de erro parece aceitável para uso prático?

### Report executivo
Inclua um mini report executivo (5–8 linhas), em linguagem de negócio, contendo:
* objetivo do modelo;
* principais achados;
* qualidade preditiva em termos simples;
* limitações;
* recomendação de uso (ou não) para apoio à decisão.

In [ ]:
# 1. Divisao dos dados usando todas as variaveis
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_all, 
    y, 
    test_size=0.25, 
    random_state=42
)

In [ ]:
# 2. Instanciacao e treinamento do modelo de Regressao Linear Multipla
multi_model = LinearRegression(fit_intercept=True)
multi_model.fit(X_train_m, y_train_m)

In [ ]:
# 3. Predicoes
y_train_pred_m = multi_model.predict(X_train_m)
y_test_pred_m = multi_model.predict(X_test_m)
residuals_m = y_test_m - y_test_pred_m

In [ ]:
# 4. Avaliacao das metricas
mae_m = mean_absolute_error(y_test_m, y_test_pred_m)
rmse_m = root_mean_squared_error(y_test_m, y_test_pred_m)
r2_train_m = r2_score(y_train_m, y_train_pred_m)
r2_test_m = r2_score(y_test_m, y_test_pred_m)

print(f"Intercepto beta_0: {multi_model.intercept_:.2f}")
print(f"R² (Treino): {r2_train_m:.3f}")
print(f"R² (Teste):  {r2_test_m:.3f}")
print(f"MAE (Teste): {mae_m:.2f}")
print(f"RMSE (Teste): {rmse_m:.2f}")

In [ ]:
# 5. Coeficientes por variavel
coef_df = pd.DataFrame({
    "Feature": X_all.columns,
    "Coeficiente": multi_model.coef_
}).sort_values(by="Coeficiente", ascending=False)

print("\nImpacto dos coeficientes:")
print(coef_df.to_string(index=False))

In [ ]:
# 6. Graficos de diagnostico
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Real vs. Previsto
axes[0].scatter(y_test_m, y_test_pred_m, alpha=0.7, color="teal")
limits = [min(y_test_m.min(), y_test_pred_m.min()), max(y_test_m.max(), y_test_pred_m.max())]
axes[0].plot(limits, limits, linestyle="dashed", color="lightgrey")
axes[0].set_xlabel("Valor real")
axes[0].set_ylabel("Valor previsto")
axes[0].set_title("Múltipla: Real vs. Previsto")

# Residuos
sns.histplot(residuals_m, kde=True, ax=axes[1], color="darkorange")
axes[1].axvline(0, color="crimson", linewidth=2)
axes[1].set_xlabel("Resíduo = real - previsto")
axes[1].set_title("Múltipla: Distribuição dos resíduos")

plt.tight_layout()
plt.show()

## Quais variáveis parecem ter maior impacto positivo e negativo no alvo?
* **Maior impacto positivo:** `s5` - concentração sérica de triglicerídeos ($\approx +695.81$), `bmi` - índice de massa corporal ($\approx +531.97$) e `s2` - colesterol ruim LDL ($\approx +508.26$). O aumento nessas variáveis está associado à aceleração da progressão da doença.

* **Maior impacto negativo:** `s1` - colesterol total ($\approx -918.50$) e `sex` ($\approx -241.99$). O valor extremo negativo de `s1` em oposição ao valor alto positivo de `s2` e `s5` reflete forte multicolinearidade entre as medições de soro sanguíneo. Matematicamente ele reduz a previsão, mas isso é apenas um efeito colateral da disputa numérica entre variáveis redundantes.

## O modelo generaliza bem do treino para o teste?
* O modelo generaliza de forma estável. O $R^2$ passou de 0.519 no treino para 0.485 no teste (uma queda modesta de apenas $\approx 0.034$ ou $3.4$ pontos percentuais), indicando que o modelo aprendeu o padrão real e não apenas decorou os dados de treino (baixo overfitting).

## Há sinais de erro sistemático (viés) nos resíduos?
* **Média e simetria:** O gráfico de sino dos erros está bem centralizado no zero. Na média geral, o estimador não apresenta viés constante para mais ou para menos.
* **Achatamento nas pontas (Real vs. Previsto):** Em casos com valores reais muito baixos (abaixo de 70), o modelo superestima; em casos muito severos (acima de 250), ele subestima. Esse comportamento decorre da regressão à média sob presença de ruído.

## O nível de erro parece aceitável para uso prático?
* **Para diagnóstico clínico crítico? Não.** A variável alvo varia de aproximadamente 25 a 346 (amplitude > 300 pontos e média próxima a 152). Como o erro médio é de 41.55 (MAE) com desvios de 53.37 (RMSE), a incerteza é elevada para decisões médicas isoladas.
* **Para triagem exploratória? Sim.** É adequado para estratificar grupos e priorizar atenção a pacientes em risco mais alto.

### Report executivo

* **Objetivo do modelo:** Estimar o índice quantitativo de progressão da diabetes após um ano com base em 10 indicadores clínicos e demográficos basais.
* **Principais achados:** O índice de massa corporal (`bmi`) e as taxas lipídicas sanguíneas (`s5`, `s2`) representam os maiores pesos positivos no avanço da doença.
* **Qualidade preditiva:** O modelo explica aproximadamente 48,5% da variabilidade no conjunto de teste, com erro médio absoluto de cerca de 41 pontos na escala de severidade.
* **Limitações:** A margem residual é ampla (RMSE = 53.37), e há perda de sensibilidade nas faixas extremas (casos muito brandos ou muito graves).
* **Recomendação de uso:** Recomendado para triagem prévia e organização de filas preventivas; não recomendado para prescrição de tratamentos clínicos sem validação complementar.